In [ ]:
import numpy as np
import random

# ------------------------
# Task and Cluster Classes
# ------------------------
class Task:
    def __init__(self, tid, criticality, wcet, period, deadline):
        self.tid = tid
        self.criticality = criticality  # 'HI' or 'LO'
        self.wcet = wcet
        self.period = period
        self.deadline = deadline
        self.utilization = wcet / period
        self.executed = False

class Cluster:
    def __init__(self, cid, num_gpus):
        self.cid = cid
        self.num_gpus = num_gpus
        self.capacity = num_gpus  # normalized capacity
        self.tasks = []

    def available_capacity(self):
        used = sum(t.utilization for t in self.tasks)
        return self.capacity - used

# ------------------------
# Q-Learning Agent
# ------------------------
class RLAgent:
    def __init__(self, state_size, action_size, alpha=0.1, gamma=0.9, epsilon=0.2):
        self.state_size = state_size
        self.action_size = action_size
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.q_table = np.zeros((state_size, action_size))

    def select_action(self, state):
        if random.random() < self.epsilon:
            return random.randint(0, self.action_size - 1)
        else:
            return np.argmax(self.q_table[state, :])

    def update_q(self, state, action, reward, next_state):
        self.q_table[state, action] += self.alpha * (
            reward + self.gamma * np.max(self.q_table[next_state, :]) - self.q_table[state, action]
        )

# ------------------------
# Scheduler
# ------------------------
class Scheduler:
    def __init__(self, clusters, agent):
        self.clusters = clusters
        self.agent = agent
        self.mode = 'LO'  # initial low-criticality mode

    def assign_priority(self, tasks):
        # DCDU: decreasing criticality, then decreasing utilization
        tasks.sort(key=lambda t: (t.criticality=='LO', -t.utilization))
        return tasks

    def allocate_tasks(self, tasks, state):
        allocation = {}
        for t in tasks:
            # Choose cluster using Worst-Fit
            best_cluster = max(self.clusters, key=lambda c: c.available_capacity())
            if t.utilization <= best_cluster.available_capacity():
                best_cluster.tasks.append(t)
                allocation[t.tid] = best_cluster.cid
            else:
                # fallback: cluster selected by RL
                action = self.agent.select_action(state)
                best_cluster = self.clusters[action % len(self.clusters)]
                best_cluster.tasks.append(t)
                allocation[t.tid] = best_cluster.cid
        return allocation

    def schedulability_test(self):
        for cluster in self.clusters:
            for t in cluster.tasks:
                if t.wcet > t.deadline:
                    return False
        return True

    def execute_tasks(self):
        # EDF: execute tasks by increasing deadline
        for cluster in self.clusters:
            cluster.tasks.sort(key=lambda t: t.deadline)
            for t in cluster.tasks:
                t.executed = True
                # Mode switch if HI task fails
                if t.criticality == 'HI' and t.wcet > t.deadline:
                    self.mode = 'HI'
                    # Drop LO tasks
                    cluster.tasks = [ti for ti in cluster.tasks if ti.criticality=='HI']

    def compute_reward(self):
        reward = 0
        hi_misses = 0
        utilization = 0
        for cluster in self.clusters:
            utilization += sum(t.utilization for t in cluster.tasks)/cluster.capacity
            for t in cluster.tasks:
                if t.criticality=='HI' and t.wcet>t.deadline:
                    hi_misses += 1
        reward += 1 if hi_misses==0 else -10*hi_misses
        reward += utilization / len(self.clusters)
        reward += 1 if self.mode=='LO' else 0
        return reward

# ------------------------
# Example Simulation
# ------------------------
if __name__ == "__main__":
    # Define clusters
    clusters = [Cluster(cid=0, num_gpus=2), Cluster(cid=1, num_gpus=2)]
    # Define agent
    agent = RLAgent(state_size=10, action_size=len(clusters))
    # Define tasks (batch B)
    tasks = [Task(tid=i, criticality='HI' if i%2==0 else 'LO', wcet=random.randint(1,3), period=5, deadline=5) for i in range(6)]
    
    scheduler = Scheduler(clusters, agent)
    
    # Priority assignment
    tasks = scheduler.assign_priority(tasks)
    print("Task Priority Queue:", [(t.tid, t.criticality, t.utilization) for t in tasks])
    
    # Allocation
    allocation = scheduler.allocate_tasks(tasks, state=0)
    print("Task Allocation:", allocation)
    
    # Schedulability test
    sched_ok = scheduler.schedulability_test()
    print("Schedulability Test Passed?", sched_ok)
    
    # Execute tasks
    scheduler.execute_tasks()
    
    # Compute reward
    reward = scheduler.compute_reward()
    print("Reward:", reward)


Task Priority Queue: [(2, 'HI', 0.6), (4, 'HI', 0.6), (0, 'HI', 0.4), (1, 'LO', 0.4), (5, 'LO', 0.4), (3, 'LO', 0.2)]
Task Allocation: {2: 0, 4: 1, 0: 0, 1: 1, 5: 0, 3: 1}
Schedulability Test Passed? True
Reward: 2.65


In [ ]:
import numpy as np
import random

# ------------------------
# Global Memory Simulation
# ------------------------
class GlobalMemory:
    def __init__(self):
        self.data = {}

    def read(self, task_id):
        return self.data.get(task_id, None)

    def write(self, task_id, value):
        self.data[task_id] = value

# ------------------------
# Task and Cluster Classes
# ------------------------
class Task:
    def __init__(self, tid, criticality, wcet, period, deadline):
        self.tid = tid
        self.criticality = criticality  # 'HI' or 'LO'
        self.wcet = wcet
        self.period = period
        self.deadline = deadline
        self.utilization = wcet / period
        self.executed = False

        # Super-block phases
        self.AP_done = False  # Acquisition Phase
        self.EP_done = False  # Execution Phase
        self.RP_done = False  # Replication Phase

class Cluster:
    def __init__(self, cid, num_gpus):
        self.cid = cid
        self.num_gpus = num_gpus
        self.capacity = num_gpus
        self.tasks = []

    def available_capacity(self):
        used = sum(t.utilization for t in self.tasks)
        return self.capacity - used

# ------------------------
# Q-Learning Agent
# ------------------------
class RLAgent:
    def __init__(self, state_size, action_size, alpha=0.1, gamma=0.9, epsilon=0.2):
        self.state_size = state_size
        self.action_size = action_size
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.q_table = np.zeros((state_size, action_size))

    def select_action(self, state):
        if random.random() < self.epsilon:
            return random.randint(0, self.action_size - 1)
        else:
            return np.argmax(self.q_table[state, :])

    def update_q(self, state, action, reward, next_state):
        self.q_table[state, action] += self.alpha * (
            reward + self.gamma * np.max(self.q_table[next_state, :]) - self.q_table[state, action]
        )

# ------------------------
# Scheduler
# ------------------------
class Scheduler:
    def __init__(self, clusters, agent, global_memory):
        self.clusters = clusters
        self.agent = agent
        self.global_memory = global_memory
        self.mode = 'LO'  # Low-criticality mode

    # ------------------------
    # Priority Assignment (DCDU)
    # ------------------------
    def assign_priority(self, tasks):
        tasks.sort(key=lambda t: (t.criticality=='LO', -t.utilization))
        return tasks

    # ------------------------
    # RL-guided Task Allocation
    # ------------------------
    def allocate_tasks(self, tasks, state):
        allocation = {}
        for t in tasks:
            best_cluster = max(self.clusters, key=lambda c: c.available_capacity())
            if t.utilization <= best_cluster.available_capacity():
                best_cluster.tasks.append(t)
                allocation[t.tid] = best_cluster.cid
            else:
                # RL fallback: choose cluster according to learned policy
                action = self.agent.select_action(state)
                best_cluster = self.clusters[action % len(self.clusters)]
                best_cluster.tasks.append(t)
                allocation[t.tid] = best_cluster.cid
        return allocation

    # ------------------------
    # Schedulability Test
    # ------------------------
    def schedulability_test(self):
        for cluster in self.clusters:
            for t in cluster.tasks:
                if t.wcet > t.deadline:
                    return False
        return True

    # ------------------------
    # Execute Tasks (Super-block Model)
    # ------------------------
    def execute_tasks(self):
        for cluster in self.clusters:
            # EDF: tasks with earliest deadlines first
            cluster.tasks.sort(key=lambda t: t.deadline)
            for t in cluster.tasks:
                # Super-block Phases
                # Acquisition Phase: read global memory
                t.AP_done = True
                self.global_memory.read(t.tid)
                
                # Execution Phase: task computation
                t.EP_done = True
                # Simulate execution (could be expanded)
                
                # Replication Phase: write results back
                t.RP_done = True
                self.global_memory.write(t.tid, f"Result_{t.tid}")
                t.executed = True

                # Check for high-criticality deadline violation
                if t.criticality == 'HI' and t.wcet > t.deadline:
                    self.mode = 'HI'
                    # Drop low-criticality tasks
                    cluster.tasks = [ti for ti in cluster.tasks if ti.criticality=='HI']

    # ------------------------
    # Reward Function
    # ------------------------
    def compute_reward(self):
        reward = 0
        hi_misses = 0
        utilization = 0
        for cluster in self.clusters:
            utilization += sum(t.utilization for t in cluster.tasks)/cluster.capacity
            for t in cluster.tasks:
                if t.criticality=='HI' and t.wcet>t.deadline:
                    hi_misses += 1
        reward += 1 if hi_misses==0 else -10*hi_misses
        reward += utilization / len(self.clusters)
        reward += 1 if self.mode=='LO' else 0
        return reward

# ------------------------
# Simulation
# ------------------------
if __name__ == "__main__":
    # Global memory
    global_mem = GlobalMemory()

    # Clusters
    clusters = [Cluster(cid=0, num_gpus=2), Cluster(cid=1, num_gpus=2)]

    # RL Agent
    agent = RLAgent(state_size=10, action_size=len(clusters))

    # Task batch
    tasks = [Task(tid=i, criticality='HI' if i%2==0 else 'LO',
                  wcet=random.randint(1,3), period=5, deadline=5) for i in range(6)]

    # Scheduler
    scheduler = Scheduler(clusters, agent, global_mem)

    # Priority assignment
    tasks = scheduler.assign_priority(tasks)
    print("Task Priority Queue:", [(t.tid, t.criticality, t.utilization) for t in tasks])

    # Task allocation
    allocation = scheduler.allocate_tasks(tasks, state=0)
    print("Task Allocation:", allocation)

    # Schedulability check
    print("Schedulability Test Passed?", scheduler.schedulability_test())

    # Execute tasks
    scheduler.execute_tasks()
    print("Executed Tasks:", [(t.tid, t.executed, t.AP_done, t.EP_done, t.RP_done) for c in clusters for t in c.tasks])

    # Compute reward
    reward = scheduler.compute_reward()
    print("Reward:", reward)


Task Priority Queue: [(0, 'HI', 0.4), (2, 'HI', 0.2), (4, 'HI', 0.2), (1, 'LO', 0.6), (3, 'LO', 0.6), (5, 'LO', 0.4)]
Task Allocation: {0: 0, 2: 1, 4: 1, 1: 0, 3: 1, 5: 0}
Schedulability Test Passed? True
Executed Tasks: [(0, True, True, True, True), (1, True, True, True, True), (5, True, True, True, True), (2, True, True, True, True), (4, True, True, True, True), (3, True, True, True, True)]
Reward: 2.6
